<a href="https://colab.research.google.com/github/ion22666/mysite/blob/main/tezaan4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Declararea functiilor MNIST

In [20]:
"""
mnist_loader
~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Incarca datele MNIST direct de pe server (nu exista date locale)
"""

import numpy as np
from tensorflow.keras.datasets import mnist

def load_data():
    (x_train, y_train), (x_test, y_test) = mnist.load_data()

    # normalizeaza (0-255 -> 0-1)
    x_train = x_train / 255.0
    x_test = x_test / 255.0

    # rearanjeaza 28x28 -> 784
    x_train = x_train.reshape(-1, 784)
    x_test = x_test.reshape(-1, 784)

    # imparte validarea (primele 50000 antrenarea, restul validare)
    training_data = (x_train[:50000], y_train[:50000])
    validation_data = (x_train[50000:], y_train[50000:])
    test_data = (x_test, y_test)

    return (training_data, validation_data, test_data)


def load_data_wrapper():
    tr_d, va_d, te_d = load_data()

    training_inputs = [np.reshape(x, (784, 1)) for x in tr_d[0]]
    training_results = [vectorized_result(y) for y in tr_d[1]]
    training_data = list(zip(training_inputs, training_results))

    validation_inputs = [np.reshape(x, (784, 1)) for x in va_d[0]]
    validation_data = list(zip(validation_inputs, va_d[1]))

    test_inputs = [np.reshape(x, (784, 1)) for x in te_d[0]]
    test_data = list(zip(test_inputs, te_d[1]))

    return (training_data, validation_data, test_data)


def vectorized_result(j):
    e = np.zeros((10, 1))
    e[j] = 1.0
    return e

# Implementarea Retelei

In [22]:
"""
Un modul pentru implementarea algoritmului de învățare prin
gradient descendent stocastic pentru o rețea neuronală feedforward.
"""

import random
import numpy as np


class Network(object):

    def __init__(self, sizes):
        """sizes = list of neurons per layer, e.g. [784, 30, 10]"""
        self.num_layers = len(sizes)
        self.sizes = sizes
        self.biases = [np.random.randn(y, 1) for y in sizes[1:]]
        self.weights = [np.random.randn(y, x)
                        for x, y in zip(sizes[:-1], sizes[1:])]

    def feedforward(self, a):
        """Return output of network for input a"""
        for b, w in zip(self.biases, self.weights):
            a = sigmoid(np.dot(w, a) + b)
        return a

    def SGD(self, training_data, epochs, mini_batch_size, eta,
            test_data=None):

        if test_data:
            test_data = list(test_data)
            n_test = len(test_data)

        training_data = list(training_data)
        n = len(training_data)

        for j in range(epochs):
            random.shuffle(training_data)

            mini_batches = [
                training_data[k:k + mini_batch_size]
                for k in range(0, n, mini_batch_size)
            ]

            for mini_batch in mini_batches:
                self.update_mini_batch(mini_batch, eta)

            if test_data:
                print(f"Epoch {j}: {self.evaluate(test_data)} / {n_test}")
            else:
                print(f"Epoch {j} complete")

    def update_mini_batch(self, mini_batch, eta):

        nabla_b = [np.zeros(b.shape) for b in self.biases]
        nabla_w = [np.zeros(w.shape) for w in self.weights]

        for x, y in mini_batch:
            delta_nabla_b, delta_nabla_w = self.backprop(x, y)

            nabla_b = [nb + dnb for nb, dnb in zip(nabla_b, delta_nabla_b)]
            nabla_w = [nw + dnw for nw, dnw in zip(nabla_w, delta_nabla_w)]

        self.weights = [
            w - (eta / len(mini_batch)) * nw
            for w, nw in zip(self.weights, nabla_w)
        ]

        self.biases = [
            b - (eta / len(mini_batch)) * nb
            for b, nb in zip(self.biases, nabla_b)
        ]

    def backprop(self, x, y):

        nabla_b = [np.zeros(b.shape) for b in self.biases]
        nabla_w = [np.zeros(w.shape) for w in self.weights]

        # forward
        activation = x
        activations = [x]
        zs = []

        for b, w in zip(self.biases, self.weights):
            z = np.dot(w, activation) + b
            zs.append(z)
            activation = sigmoid(z)
            activations.append(activation)

        # backward
        delta = self.cost_derivative(activations[-1], y) * sigmoid_prime(zs[-1])
        nabla_b[-1] = delta
        nabla_w[-1] = np.dot(delta, activations[-2].T)

        for l in range(2, self.num_layers):
            z = zs[-l]
            sp = sigmoid_prime(z)
            delta = np.dot(self.weights[-l + 1].T, delta) * sp

            nabla_b[-l] = delta
            nabla_w[-l] = np.dot(delta, activations[-l - 1].T)

        return (nabla_b, nabla_w)

    def evaluate(self, test_data):

        test_results = [
            (np.argmax(self.feedforward(x)), y)
            for (x, y) in test_data
        ]

        return sum(int(x == y) for (x, y) in test_results)

    def cost_derivative(self, output_activations, y):
        return (output_activations - y)


# ---- helpers ----

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def sigmoid_prime(z):
    return sigmoid(z) * (1 - sigmoid(z))

# Invatarea retelei

1. Încărcarea datelor MNIST

In [28]:
# încărcăm datele (din versiunea fără fișiere locale)
training_data, validation_data, test_data = load_data_wrapper()

2. Inițializarea rețelei
Pornim cu o rețea simplă:

*   784 neuroni de intrare (28×28 pixeli)
*   30 neuroni în stratul ascuns
*   10 neuroni la ieșire (cifrele 0-9)







In [33]:
net = Network([784, 30, 10])

3. Antrenarea rețelei (SGD)

Folosim:
* 30 epoci
* mini-batch = 10
* rata de învățare η = 3.0

In [54]:
net.SGD(training_data, epochs=30, mini_batch_size=10, eta=3.0, test_data=test_data)

Epoch 0: 9572 / 10000
Epoch 1: 9582 / 10000
Epoch 2: 9587 / 10000
Epoch 3: 9600 / 10000
Epoch 4: 9608 / 10000
Epoch 5: 9614 / 10000
Epoch 6: 9609 / 10000
Epoch 7: 9614 / 10000
Epoch 8: 9611 / 10000


KeyboardInterrupt: 

4. Îmbunătățirea modelului

Dacă creștem numărul de neuroni ascunși:

In [39]:
net = Network([784, 100, 10])
net.SGD(training_data, 30, 10, 3.0, test_data=test_data)

Epoch 0: 8226 / 10000
Epoch 1: 9297 / 10000
Epoch 2: 9387 / 10000
Epoch 3: 9445 / 10000
Epoch 4: 9507 / 10000
Epoch 5: 9515 / 10000
Epoch 6: 9554 / 10000
Epoch 7: 9534 / 10000
Epoch 8: 9567 / 10000
Epoch 9: 9569 / 10000
Epoch 10: 9565 / 10000


KeyboardInterrupt: 

5. Importanța hiperparametrilor
Learning rate prea mic

In [25]:
net = Network([784, 30, 10])
net.SGD(training_data, 30, 10, 100.0, test_data=test_data)

Epoch 0: 974 / 10000
Epoch 1: 974 / 10000
Epoch 2: 974 / 10000
Epoch 3: 974 / 10000


KeyboardInterrupt: 

# Testarea

In [53]:
import base64
import numpy as np
import cv2
from IPython.display import display
from google.colab import output

def predict_digit(data_url):
    # decode base64
    header, encoded = data_url.split(",", 1)
    img_bytes = base64.b64decode(encoded)

    # convert to numpy
    nparr = np.frombuffer(img_bytes, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_GRAYSCALE)

    # resize la 28x28 (MNIST)
    img = cv2.resize(img, (28, 28))

    # blur → introduce valori de gri (IMPORTANT
    img = cv2.GaussianBlur(img, (5, 5), 0)

    # invert (MNIST = alb pe negru)
    img = 255 - img

    # normalize
    img = img / 255.0

    # reshape
    img = img.reshape(784, 1)

    # predict
    output_vals = net.feedforward(img)
    prediction = np.argmax(output_vals)

    print("Predicție:", prediction)

# legăm JS de Python
output.register_callback('notebook.predict_digit', predict_digit)

In [57]:
from IPython.display import display, HTML

display(HTML("""
<canvas id="canvas" width="280" height="280" style="border:1px solid black"></canvas>
<br>
<button onclick="clearCanvas()">Clear</button>
<button onclick="predict()">Predict</button>

<script>
var canvas = document.getElementById('canvas');
var ctx = canvas.getContext('2d');

ctx.fillStyle = "black";
ctx.fillRect(0, 0, canvas.width, canvas.height);

var drawing = false;

canvas.onmousedown = function(e) {
    drawing = true;
}

canvas.onmouseup = function(e) {
    drawing = false;
    ctx.beginPath();
}

canvas.onmousemove = function(e) {
    if (!drawing) return;
    ctx.lineWidth = 15;
    ctx.lineCap = 'round';
    ctx.strokeStyle = 'white';

    ctx.lineTo(e.offsetX, e.offsetY);
    ctx.stroke();
    ctx.beginPath();
    ctx.moveTo(e.offsetX, e.offsetY);
}

function clearCanvas() {
    ctx.clearRect(0, 0, canvas.width, canvas.height); // șterge complet
    ctx.fillStyle = "black";
    ctx.fillRect(0, 0, canvas.width, canvas.height);
    ctx.beginPath(); // 🔥 reset traseu
}

function predict() {
    var dataURL = canvas.toDataURL();
    google.colab.kernel.invokeFunction('notebook.predict_digit', [dataURL], {});
}
</script>
"""))

Predicție: 2
Predicție: 2
Predicție: 8
Predicție: 6
Predicție: 8
